In [2]:

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42
TARGET = "int_rate"
ID_COL = "ID"

# -----------------------------
# 1. LOAD DATA
# -----------------------------

if not os.path.exists("LC_train.csv"):
    raise FileNotFoundError("Upload LC_train.csv to Colab first.")

if not os.path.exists("LC_test.csv"):
    raise FileNotFoundError("Upload LC_test.csv to Colab first.")

train = pd.read_csv("LC_train.csv", na_values=["NA"])
test = pd.read_csv("LC_test.csv", na_values=["NA"])

print("Original train:", train.shape)
print("Original test:", test.shape)

if TARGET not in train.columns:
    raise ValueError(f"Target column {TARGET} not found. Available columns: {train.columns.tolist()}")

train[TARGET] = pd.to_numeric(train[TARGET], errors="coerce")
train = train.dropna(subset=[TARGET]).copy()

# -----------------------------
# 2. DROP LEAKAGE COLUMNS
# -----------------------------

leakage_cols = [
    "loan_status", "grade", "sub_grade", "installment",
    "last_fico_range_high", "last_fico_range_low",
    "last_pymnt_amnt", "last_pymnt_d", "next_pymnt_d",
    "recoveries", "collection_recovery_fee",
    "total_pymnt", "total_pymnt_inv",
    "total_rec_prncp", "total_rec_int", "total_rec_late_fee",
    "out_prncp", "out_prncp_inv",
    "hardship_flag", "debt_settlement_flag"
]

train = train.drop(columns=[c for c in leakage_cols if c in train.columns])
test = test.drop(columns=[c for c in leakage_cols if c in test.columns])

print("After leakage removal train:", train.shape)
print("After leakage removal test:", test.shape)

# -----------------------------
# 3. FEATURE ENGINEERING
# -----------------------------

def feature_engineering(df):
    df = df.copy()

    if "fico_range_low" in df.columns and "fico_range_high" in df.columns:
        df["fico_mid"] = (df["fico_range_low"] + df["fico_range_high"]) / 2
        df = df.drop(columns=["fico_range_low", "fico_range_high"])

    if "term" in df.columns:
        df["term_months"] = df["term"].astype(str).str.extract(r"(\d+)", expand=False).astype(float)

    if "emp_length" in df.columns:
        emp_map = {
            "< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3,
            "4 years": 4, "5 years": 5, "6 years": 6, "7 years": 7,
            "8 years": 8, "9 years": 9, "10+ years": 10
        }
        df["emp_length_num"] = df["emp_length"].map(emp_map)

    if "loan_amnt" in df.columns and "annual_inc" in df.columns:
        df["loan_to_income"] = df["loan_amnt"] / (df["annual_inc"] + 1)

    if "dti" in df.columns and "annual_inc" in df.columns:
        df["dti_income_interaction"] = df["dti"] * np.log1p(df["annual_inc"].clip(lower=0))

    if "fico_mid" in df.columns and "revol_util" in df.columns:
        df["fico_revol_interaction"] = df["fico_mid"] * df["revol_util"]

    if "fico_mid" in df.columns and "dti" in df.columns:
        df["fico_dti_interaction"] = df["fico_mid"] * df["dti"]

    log_cols = ["annual_inc", "loan_amnt", "revol_bal", "tot_cur_bal", "total_bal_ex_mort", "tot_coll_amt"]

    for col in log_cols:
        if col in df.columns:
            df[col + "_log"] = np.log1p(df[col].clip(lower=0))

    missing_cols = ["mths_since_last_record", "mths_since_recent_inq", "mths_since_rcnt_il", "mths_since_recent_bc"]

    for col in missing_cols:
        if col in df.columns:
            df[col + "_missing"] = df[col].isna().astype(int)

    return df

train_fe = feature_engineering(train)
test_fe = feature_engineering(test)

# -----------------------------
# 4. SPLIT DATA
# -----------------------------

X = train_fe.drop(columns=[TARGET])
y = train_fe[TARGET]

if ID_COL in test_fe.columns:
    test_ids = test_fe[ID_COL]
    X_test = test_fe.drop(columns=[ID_COL])
else:
    test_ids = pd.Series(range(len(test_fe)), name=ID_COL)
    X_test = test_fe.copy()

X_test = X_test.reindex(columns=X.columns, fill_value=np.nan)

X = X.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

# -----------------------------
# 5. PREPROCESSORS
# -----------------------------

onehot_preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))
    ]), categorical_features)
])

linear_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False))
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))
    ]), categorical_features)
])

ordinal_preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
    ]), categorical_features)
])

# -----------------------------
# 6. MODELS
# -----------------------------

models = {
    "Random Forest": Pipeline([
        ("preprocessor", onehot_preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=250,
            max_depth=25,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features=0.7,
            random_state=RANDOM_STATE,
            n_jobs=1
        ))
    ]),

    "Extra Trees": Pipeline([
        ("preprocessor", onehot_preprocessor),
        ("model", ExtraTreesRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features=0.7,
            random_state=RANDOM_STATE,
            n_jobs=1
        ))
    ]),

    "Ridge": Pipeline([
        ("preprocessor", linear_preprocessor),
        ("model", RidgeCV(alphas=[0.1, 1, 10, 30, 100, 300]))
    ]),

    "Hist Gradient Boosting": Pipeline([
        ("preprocessor", ordinal_preprocessor),
        ("model", HistGradientBoostingRegressor(
            max_iter=250,
            learning_rate=0.05,
            max_leaf_nodes=31,
            l2_regularization=0.1,
            random_state=RANDOM_STATE
        ))
    ])
}

valid_predictions = {}
test_predictions = {}
rows = []

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)

    valid_pred = model.predict(X_valid)
    test_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))
    mae = mean_absolute_error(y_valid, valid_pred)
    r2 = r2_score(y_valid, valid_pred)

    valid_predictions[name] = valid_pred
    test_predictions[name] = test_pred

    rows.append({
        "Model": name,
        "Validation RMSE": rmse,
        "Validation MAE": mae,
        "Validation R2": r2
    })

    print(name, "RMSE:", rmse)

results = pd.DataFrame(rows).sort_values("Validation RMSE")
display(results)

# -----------------------------
# 7. BLEND MODELS TO LOWER RMSE
# -----------------------------

blend_results = []

model_names = list(valid_predictions.keys())

for rf_w in np.arange(0, 1.01, 0.1):
    for et_w in np.arange(0, 1.01 - rf_w, 0.1):
        for ridge_w in np.arange(0, 1.01 - rf_w - et_w, 0.1):
            hgb_w = 1 - rf_w - et_w - ridge_w

            if hgb_w < -0.0001:
                continue

            blend_pred = (
                rf_w * valid_predictions["Random Forest"] +
                et_w * valid_predictions["Extra Trees"] +
                ridge_w * valid_predictions["Ridge"] +
                hgb_w * valid_predictions["Hist Gradient Boosting"]
            )

            rmse = np.sqrt(mean_squared_error(y_valid, blend_pred))
            mae = mean_absolute_error(y_valid, blend_pred)
            r2 = r2_score(y_valid, blend_pred)

            blend_results.append({
                "Random Forest Weight": rf_w,
                "Extra Trees Weight": et_w,
                "Ridge Weight": ridge_w,
                "Hist Gradient Boosting Weight": hgb_w,
                "RMSE": rmse,
                "MAE": mae,
                "R2": r2
            })

blend_results = pd.DataFrame(blend_results).sort_values("RMSE")
display(blend_results.head(10))

best_blend = blend_results.iloc[0]

print("\nBest Blend:")
print(best_blend)

# -----------------------------
# 8. FINAL TEST SUBMISSION
# -----------------------------

final_test_pred = (
    best_blend["Random Forest Weight"] * test_predictions["Random Forest"] +
    best_blend["Extra Trees Weight"] * test_predictions["Extra Trees"] +
    best_blend["Ridge Weight"] * test_predictions["Ridge"] +
    best_blend["Hist Gradient Boosting Weight"] * test_predictions["Hist Gradient Boosting"]
)

final_test_pred = np.clip(final_test_pred, 0, 100)

submission = pd.DataFrame({
    ID_COL: test_ids,
    "int_rate": final_test_pred
})

submission.to_csv("low_rmse_submission.csv", index=False)
results.to_csv("model_results.csv", index=False)
blend_results.to_csv("blend_results.csv", index=False)

print("\nSaved files:")
print("low_rmse_submission.csv")
print("model_results.csv")
print("blend_results.csv")

display(submission.head())

Original train: (19519, 39)
Original test: (10000, 39)
After leakage removal train: (19518, 38)
After leakage removal test: (10000, 38)

Training Random Forest...
Random Forest RMSE: 3.928732777833347

Training Extra Trees...
Extra Trees RMSE: 3.9879458177109197

Training Ridge...
Ridge RMSE: 4.118876083502155

Training Hist Gradient Boosting...
Hist Gradient Boosting RMSE: 3.8335652505824114


,Model,Validation RMSE,Validation MAE,Validation R2
3,Hist Gradient Boosting,3.833565,2.848446,0.444218
0,Random Forest,3.928733,2.912798,0.416281
1,Extra Trees,3.987946,2.934871,0.398554
2,Ridge,4.118876,3.158628,0.358412


,Random Forest Weight,Extra Trees Weight,Ridge Weight,Hist Gradient Boosting Weight,RMSE,MAE,R2
11,0.0,0.1,0.0,0.9,3.831918,2.844520,0.444696
66,0.1,0.0,0.0,0.9,3.832054,2.846917,0.444656
67,0.1,0.0,0.1,0.8,3.832412,2.852087,0.444553
76,0.1,0.1,0.0,0.8,3.832434,2.843589,0.444546
1,0.0,0.0,0.1,0.9,3.832722,2.853521,0.444463
121,0.2,0.0,0.0,0.8,3.833020,2.846429,0.444376
12,0.0,0.1,0.1,0.8,3.833241,2.850617,0.444312
0,0.0,0.0,0.0,1.0,3.833565,2.848446,0.444218
21,0.0,0.2,0.0,0.8,3.834138,2.843103,0.444052
122,0.2,0.0,0.1,0.7,3.834579,2.852518,0.443924



Best Blend:
Random Forest Weight             0.000000
Extra Trees Weight               0.100000
Ridge Weight                     0.000000
Hist Gradient Boosting Weight    0.900000
RMSE                             3.831918
MAE                              2.844520
R2                               0.444696
Name: 11, dtype: float64

Saved files:
low_rmse_submission.csv
model_results.csv
blend_results.csv


,ID,int_rate
0,1,12.166350
1,2,13.612548
2,3,7.576761
3,4,17.506332
4,5,14.753410
